In [ ]:
import torch
import torch.nn as nn

In [ ]:
# torch.backends.cudnn.benchmark = True
# this thing is not working good instead of decreasing this is increasing the time.

In [ ]:
import os
print(os.cpu_count())

2


In [ ]:
!unzip -q /content/drive/MyDrive/DDPM/train.zip -d /content/data

In [ ]:
!ls /content/data/train/images/0 | wc -l

5923


In [ ]:
class NoiseScheduler:

  def __init__(self, num_timesteps, beta_start, beta_end, device):
    self.num_timesteps = num_timesteps
    self.beta_start = beta_start
    self.beta_end = beta_end
    self.device = device
    self.betas = torch.linspace(beta_start, beta_end, num_timesteps).to(self.device)
    self.alphas = 1-self.betas
    self.alphas_cum_prod = torch.cumprod(self.alphas, dim=0).to(self.device)
    self.sqrt_alphas_cum_prod = torch.sqrt(self.alphas_cum_prod).to(self.device)
    self.one_minus_alphas_cum_prod = 1 - self.alphas_cum_prod # Define the missing attribute
    self.sqrt_one_minus_alphas_cum_prod = torch.sqrt(self.one_minus_alphas_cum_prod).to(self.device)

  def add_noise(self, original, noise, t):
    original_shape = original.shape
    batch_size = original_shape[0]
    sqrt_alphas_cum_prod = self.sqrt_alphas_cum_prod[t].reshape(batch_size)
    sqrt_one_minus_alphas_cum_prod = self.sqrt_one_minus_alphas_cum_prod[t].reshape(batch_size)

    for _ in range(len(original_shape)-1):
      sqrt_alphas_cum_prod = sqrt_alphas_cum_prod.unsqueeze(-1)
      sqrt_one_minus_alphas_cum_prod = sqrt_one_minus_alphas_cum_prod.unsqueeze(-1)

    return sqrt_alphas_cum_prod * original + sqrt_one_minus_alphas_cum_prod * noise

  def sample_prev_timestep(self, xt, noise_pred, t):
    x0 = (xt - noise_pred * self.sqrt_one_minus_alphas_cum_prod[t]) / self.sqrt_alphas_cum_prod[t]
    x0 = torch.clamp(x0, -1., 1.)

    mean = (xt - self.betas[t]*noise_pred/self.sqrt_one_minus_alphas_cum_prod[t])
    mean = mean/torch.sqrt(self.alphas[t])

    if(t==0):
      return mean, x0
    else:
      variance = (1-self.alphas[t])*(self.one_minus_alphas_cum_prod[t-1])/self.one_minus_alphas_cum_prod[t]
      sigma = torch.sqrt(variance)
      z=torch.randn(xt.shape).to(xt.device)
      return mean + sigma*z, x0

In [ ]:
def get_time_embeddings(time_steps, t_emb_dim):
  factor = 10000**(torch.arange(0,t_emb_dim//2, device = time_steps.device)//(t_emb_dim//2))
  t_emb = time_steps[:, None].repeat(1, t_emb_dim//2)/ factor
  t_emb = torch.cat((torch.sin(t_emb), torch.cos(t_emb)), dim=-1)
  return t_emb

class DownBlock(nn.Module):
  def __init__(self, in_channels, out_channels, t_emb_dim,
               down_sample, num_heads):
    super().__init__()
    self.down_sample = down_sample
    self.resnet_conv_first = nn.Sequential(
        nn.GroupNorm(8, in_channels),
        nn.SiLU(),
        nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1)
        )
    self.time_emb = nn.Sequential(
        nn.SiLU(),
        nn.Linear(t_emb_dim, out_channels)
    )

    self.resnet_conv_second = nn.Sequential(
        nn.GroupNorm(8, out_channels),
        nn.SiLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
    )

    self.attention_norm = nn.GroupNorm(8, out_channels)
    self.attention = nn.MultiheadAttention(out_channels, num_heads, batch_first=True)
    self.residual_input_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)
    self.down_sample_conv = nn.Conv2d(out_channels, out_channels, kernel_size=4,
                                      stride=2, padding=1) if self.down_sample else nn.Identity()

  def forward(self, x, t_emb):
    out = x
    # Resnet block
    resnet_input = out
    out = self.resnet_conv_first(out)
    out = out + self.time_emb(t_emb)[:,:,None,None]
    out = self.resnet_conv_second(out)
    out = out + self.residual_input_conv(resnet_input)

    # Attention block
    batch_size, channels, h, w = out.shape
    in_attn = out.reshape(batch_size, channels, h*w)
    in_attn = self.attention_norm(in_attn)
    in_attn = in_attn.transpose(1,2)
    out_attn, _ = self.attention(in_attn, in_attn, in_attn)
    out_attn = out_attn.transpose(1,2).reshape(batch_size, channels, h, w)
    out = out + out_attn

    out = self.down_sample_conv(out)
    return out


In [ ]:
class MidBlock(nn.Module):
  def __init__(self, in_channels, out_channels, t_emb_dim, num_heads):
    super().__init__()
    self.resnet_conv_first = nn.ModuleList([
        nn.Sequential(
            nn.GroupNorm(8, in_channels),
            nn.SiLU(),
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1)),
        nn.Sequential(
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1))
    ])
    self.time_emb = nn.ModuleList([
        nn.Sequential(
            nn.SiLU(),
            nn.Linear(t_emb_dim, out_channels)),
        nn.Sequential(
            nn.SiLU(),
            nn.Linear(t_emb_dim, out_channels))
    ])

    self.resnet_conv_second = nn.ModuleList([
        nn.Sequential(
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        ),
        nn.Sequential(
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        )
    ])

    self.attention_norm = nn.GroupNorm(8, out_channels)
    self.attention = nn.MultiheadAttention(out_channels, num_heads, batch_first=True)
    self.residual_input_conv = nn.ModuleList([
        nn.Conv2d(in_channels, out_channels, kernel_size=1),
        nn.Conv2d(out_channels, out_channels, kernel_size=1)
    ])

  def forward(self, x, t_emb):
    out = x
    # Resnet block
    resnet_input = out
    out = self.resnet_conv_first[0](out)
    out = out + self.time_emb[0](t_emb)[:,:,None,None]
    out = self.resnet_conv_second[0](out)
    out = out + self.residual_input_conv[0](resnet_input)

    # Attention block
    batch_size, channels, h, w = out.shape
    in_attn = out.reshape(batch_size, channels, h*w)
    in_attn = self.attention_norm(in_attn)
    in_attn = in_attn.transpose(1,2)
    out_attn, _ = self.attention(in_attn, in_attn, in_attn)
    out_attn = out_attn.transpose(1,2).reshape(batch_size, channels, h, w)
    out = out + out_attn

    # Second Resnet block
    resnet_input = out
    out = self.resnet_conv_first[1](out)
    out = out + self.time_emb[1](t_emb)[:,:,None,None]
    out = self.resnet_conv_second[1](out)
    out = out + self.residual_input_conv[1](resnet_input)

    return out

In [ ]:
class UpBlock(nn.Module):
  def __init__(self, in_channels, out_channels, t_emb_dim, up_sample, num_heads):
    super().__init__()
    self.up_sample = up_sample
    # self.num_heads = num_heads
    self.resnet_conv_first = nn.Sequential(
        nn.GroupNorm(8, in_channels),
        nn.SiLU(),
        nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1)
        )
    self.time_emb = nn.Sequential(
        nn.SiLU(),
        nn.Linear(t_emb_dim, out_channels)
    )

    self.resnet_conv_second = nn.Sequential(
        nn.GroupNorm(8, out_channels),
        nn.SiLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
    )

    self.attention_norm = nn.GroupNorm(8, out_channels)
    self.attention = nn.MultiheadAttention(out_channels, num_heads, batch_first=True)
    self.residual_input_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)
    self.up_sample_conv = nn.ConvTranspose2d(in_channels//2, in_channels//2, kernel_size=4,
                                      stride=2, padding=1) if self.up_sample else nn.Identity()

  def forward(self, x, out_down, t_emb):
    x=self.up_sample_conv(x)
    x= torch.cat((x, out_down), dim=1)

    out = x
    # Resnet block
    resnet_input = out
    out = self.resnet_conv_first(out)
    out = out + self.time_emb(t_emb)[:,:,None,None]
    out = self.resnet_conv_second(out)
    out = out + self.residual_input_conv(resnet_input)

    # Attention block
    batch_size, channels, h, w = out.shape
    in_attn = out.reshape(batch_size, channels, h*w)
    in_attn = self.attention_norm(in_attn)
    in_attn = in_attn.transpose(1,2)
    out_attn, _ = self.attention(in_attn, in_attn, in_attn)
    out_attn = out_attn.transpose(1,2).reshape(batch_size, channels, h, w)
    out = out + out_attn

    return out



In [ ]:
class Unet(nn.Module):
  def __init__(self, im_channels):
    super().__init__()
    self.down_channels = [32, 64, 128, 256]
    self.mid_channels = [256, 256, 128]
    self.t_emb_dim = 128
    self.down_sample = [True, True, False]

    self.t_proj = nn.Sequential(
        nn.Linear(self.t_emb_dim, self.t_emb_dim),
        nn.SiLU(),
        nn.Linear(self.t_emb_dim, self.t_emb_dim)
    )

    self.up_channels = self.down_channels[::-1]
    self.conv_in = nn.Conv2d(im_channels, self.down_channels[0], kernel_size=3, padding=1)

    self.down = nn.ModuleList([])
    for i in range(len(self.down_channels)-1):
      self.down.append(DownBlock(self.down_channels[i], self.down_channels[i+1], self.t_emb_dim,
                                 down_sample=self.down_sample[i], num_heads=4))

    self.mid = nn.ModuleList([])
    for i in range(len(self.mid_channels)-1):
      self.mid.append(MidBlock(self.mid_channels[i], self.mid_channels[i+1], self.t_emb_dim, num_heads=4))

    self.ups = nn.ModuleList([])
    for i in reversed(range(len(self.down_channels)-1)):
      self.ups.append(UpBlock(self.down_channels[i]*2, self.down_channels[i-1] if i!=0 else 16,
                              self.t_emb_dim, up_sample=self.down_sample[i], num_heads=4))

    self.norm_out = nn.GroupNorm(8, 16)
    self.conv_out = nn.Conv2d(16, im_channels, kernel_size=3, padding=1)

  def forward(self, x, t):
    out = self.conv_in(x)
    t_emb = get_time_embeddings(t, self.t_emb_dim)
    t_emb = self.t_proj(t_emb)

    down_outs=[]

    # commenting the print lines they print the model shape or the output shape for every pass
    for down in self.down:
      # print(out.shape)
      down_outs.append(out)
      out = down(out, t_emb)

    for mid in self.mid:
      # print(out.shape)
      out = mid(out, t_emb)

    for up in self.ups:
      down_out = down_outs.pop()
      # print(out.shape)
      # print(down_out.shape)
      out = up(out, down_out, t_emb)

    out = self.norm_out(out)
    out = nn.SiLU()(out)
    out = self.conv_out(out)
    return out

mnist dataset.py

In [ ]:
import glob
import os

import torchvision
from PIL import Image
from tqdm import tqdm
from torch.utils.data.dataloader import DataLoader
from torch.utils.data.dataset import Dataset


class MnistDataset(Dataset):
    r"""
    Nothing special here. Just a simple dataset class for mnist images.
    Created a dataset class rather using torchvision to allow
    replacement with any other image dataset
    """
    def __init__(self, split, im_path, im_ext='png'):
        r"""
        Init method for initializing the dataset properties
        :param split: train/test to locate the image files
        :param im_path: root folder of images
        :param im_ext: image extension. assumes all
        images would be this type.
        """
        self.split = split
        self.im_ext = im_ext
        self.images, self.labels = self.load_images(im_path)

    def load_images(self, im_path):
        r"""
        Gets all images from the path specified
        and stacks them all up
        :param im_path:
        :return:
        """
        assert os.path.exists(im_path), "images path {} does not exist".format(im_path)
        ims = []
        labels = []
        to_tensor = torchvision.transforms.ToTensor() # new code
        for d_name in tqdm(os.listdir(im_path)):
            for fname in glob.glob(os.path.join(im_path, d_name, '*.{}'.format(self.im_ext))):
                im = Image.open(fname)
                im_tensor = to_tensor(im)
                im_tensor = (2 * im_tensor) - 1
                ims.append(im_tensor)
                labels.append(int(d_name))
                # ims.append(fname)  # this is the previous code
                # labels.append(int(d_name))
        print('Found {} images for split {}'.format(len(ims), self.split))
        return ims, labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        # this was the previous code
        # im = Image.open(self.images[index])
        # im_tensor = torchvision.transforms.ToTensor()(im)

        # # Convert input to -1 to 1 range.
        # im_tensor = (2 * im_tensor) - 1
        # return im_tensor
        return self.images[index]

In [ ]:
%%writefile config.yaml
dataset_params:
  im_path: '/content/data/train/images'

diffusion_params:
  num_timesteps: 1000
  beta_start: 0.0001
  beta_end: 0.02

model_params:
  im_channels: 1
  im_size: 28
  down_channels: [32, 64, 128, 256]
  mid_channels: [256, 256, 128]
  down_sample: [True, True, False]
  time_emb_dim: 128
  num_down_layers: 2
  num_mid_layers: 2
  num_up_layers: 2
  num_heads: 4

train_params:
  task_name: 'default'
  batch_size: 192 #changes to 128 from 64. Changing again to 192 from 128. changing it from 192 to 384
  num_epochs: 40
  num_samples: 100
  num_grid_rows: 10
  lr: 0.0001
  ckpt_name: 'ddpm_ckpt.pth'

Writing config.yaml


In [ ]:
!mkdir default

Sample DDPM

In [ ]:
# import torch
import torchvision
import argparse
import yaml
import os
from torchvision.utils import make_grid
from tqdm import tqdm
# from models.unet_base import Unet
# from scheduler.linear_noise_scheduler import LinearNoiseScheduler


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


from torch.cuda.amp import autocast # Import AMP for fast inference

def sample(model, scheduler, train_config, model_config, diffusion_config):
    r"""
    Sample stepwise by going backward one timestep at a time.
    """
    # Create the samples folder ONCE before the loop starts
    save_dir = os.path.join(train_config['task_name'], 'samples')
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    # Initialize pure noise
    xt = torch.randn((train_config['num_samples'],
                      model_config['im_channels'],
                      model_config['im_size'],
                      model_config['im_size'])).to(device)

    for i in tqdm(reversed(range(diffusion_config['num_timesteps']))):

        # Create a timestep tensor that matches the batch size!
        t_tensor = torch.full((train_config['num_samples'],), i, device=device, dtype=torch.long)

        # Try Use 16-bit math for faster inference. Results were not much decreased but time taken was less
        # with autocast():
        # this is currently without autocast
        noise_pred = model(xt, t_tensor)

        # Use scheduler to get xt-1
        # (Assuming your scheduler expects the same batched t_tensor)
        # xt, x0_pred = scheduler.sample_prev_timestep(xt, noise_pred, t_tensor)
        xt, x0_pred = scheduler.sample_prev_timestep(xt, noise_pred, torch.as_tensor(i).to(device))
        # save images every 100 steps AND on the final step (0)
        if i % 100 == 0 or i == 0:
            ims = torch.clamp(xt, -1., 1.).detach().cpu()
            ims = (ims + 1) / 2
            grid = make_grid(ims, nrow=train_config['num_grid_rows'])
            img = torchvision.transforms.ToPILImage()(grid)

            # Save the image
            img.save(os.path.join(save_dir, 'xt_{}.png'.format(i)))
            img.close()
        # ONLY run this on the absolute final step when the image is perfect
        if i == 0:
            # 1. Create a dedicated folder for these 100 images
            individual_dir = os.path.join(train_config['task_name'], 'samples', 'individual_digits')
            if not os.path.exists(individual_dir):
                os.makedirs(individual_dir)

            # 2. Loop through your tensor of 100 digits and save them individually
            for batch_idx in range(ims.shape[0]):
                single_img_tensor = ims[batch_idx]
                pil_img = torchvision.transforms.ToPILImage()(single_img_tensor)

                # Saves as digit_000.png, digit_001.png, etc.
                pil_img.save(os.path.join(individual_dir, f'digit_{batch_idx:03d}.png'))
                pil_img.close()


def infer(args):
    # Read the config file
    with open(args.config_path, 'r') as file:
        try:
            config = yaml.safe_load(file)
        except yaml.YAMLError as exc:
            print(exc)

    diffusion_config = config['diffusion_params']
    model_config = config['model_params']
    train_config = config['train_params']

    # Initialize the raw model
    model = Unet(model_config['im_channels']).to(device)

    # Clean the saved weights before loading
    state_dict = torch.load(os.path.join(train_config['task_name'], train_config['ckpt_name']), map_location=device)

    # Strip out the '_orig_mod.' prefix that torch.compile added
    clean_state_dict = {}
    for key, value in state_dict.items():
        if key.startswith('_orig_mod.'):
            clean_state_dict[key.replace('_orig_mod.', '')] = value
        else:
            clean_state_dict[key] = value

    # Load the cleaned weights
    model.load_state_dict(clean_state_dict)
    model.eval()


    scheduler = NoiseScheduler(num_timesteps=diffusion_config['num_timesteps'],
                                     beta_start=diffusion_config['beta_start'],
                                     beta_end=diffusion_config['beta_end'],
                                     device=device)

    # Run the sampling
    with torch.no_grad():
        sample(model, scheduler, train_config, model_config, diffusion_config)

if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='Arguments for ddpm image generation')
    parser.add_argument('--config', dest='config_path',
                        default='/content/config.yaml', type=str)
    args = parser.parse_args([]) # Modified this line to handle Colab kernel arguments
    infer(args)

1000it [01:58,  8.47it/s]


In [ ]:
import shutil
import os

# 1. Define the path to your samples folder
folder_to_zip = '/content/default/samples/individual_digits'

# 2. Define what you want the zip file to be named
output_filename = 'ddpm_generated_images_single'

# 3. Create the zip file
shutil.make_archive(output_filename, 'zip', folder_to_zip)

print(f"Successfully zipped! You can now download: {output_filename}.zip")

Successfully zipped! You can now download: ddpm_generated_images_single.zip


Train DDPM

In [ ]:
# import torch
import yaml
import argparse
import os
import numpy as np
from tqdm import tqdm
from torch.optim import Adam
# from dataset.mnist_dataset import MnistDataset
from torch.utils.data import DataLoader
# from models.unet_base import Unet
# from scheduler.linear_noise_scheduler import LinearNoiseScheduler

# Import the AMP tools
from torch.cuda.amp import GradScaler, autocast

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def train(args):
    # Read the config file #
    with open(args.config_path, 'r') as file:
        try:
            config = yaml.safe_load(file)
        except yaml.YAMLError as exc:
            print(exc)
    print(config)
    ########################

    diffusion_config = config['diffusion_params']
    dataset_config = config['dataset_params']
    model_config = config['model_params']
    train_config = config['train_params']

    # Create the noise scheduler
    scheduler = NoiseScheduler(num_timesteps=diffusion_config['num_timesteps'],
                                     beta_start=diffusion_config['beta_start'],
                                     beta_end=diffusion_config['beta_end'],
                                     device=device)

    # Create the dataset
    mnist = MnistDataset('train', im_path=dataset_config['im_path'])
    mnist_loader = DataLoader(mnist, batch_size=train_config['batch_size'], shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    # mnist_loader = DataLoader(mnist, batch_size=train_config['batch_size'], shuffle=True, num_workers=4) # previous code
    # Instantiate the model
    model = Unet(model_config['im_channels']).to(device)
    model = torch.compile(model) # Will use this thing as a master stroke
    model.train()

    # Create output directories
    if not os.path.exists(train_config['task_name']):
        os.mkdir(train_config['task_name'])

    # Load checkpoint if found
    if os.path.exists(os.path.join(train_config['task_name'],train_config['ckpt_name'])):
        print('Loading checkpoint as found one')
        model.load_state_dict(torch.load(os.path.join(train_config['task_name'],
                                                      train_config['ckpt_name']), map_location=device))
    # Specify training parameters
    num_epochs = train_config['num_epochs']
    optimizer = Adam(model.parameters(), lr=train_config['lr'])
    criterion = torch.nn.MSELoss()

    # Initialize the Scaler right BEFORE your training loop begins
    scaler = GradScaler()

    # Run training
    for epoch_idx in range(num_epochs):
        losses = []
        for im in tqdm(mnist_loader):
            optimizer.zero_grad()
            im = im.float().to(device, non_blocking=True)
            # im = im.float().to(device) # previous code
            # Sample random noise
            noise = torch.randn_like(im).to(device)

            # Sample timestep
            t = torch.randint(0, diffusion_config['num_timesteps'], (im.shape[0],)).to(device)

            with autocast():
              # Add noise to images according to timestep
              noisy_im = scheduler.add_noise(im, noise, t)
              noise_pred = model(noisy_im, t)
              loss = criterion(noise_pred, noise)

            losses.append(loss.item())
            # loss.backward()
            # optimizer.step()

            # Scale the loss and do the backward pass
            scaler.scale(loss).backward()

            # Step the optimizer and update the scaler for the next batch
            scaler.step(optimizer)
            scaler.update()
            # --- AMP END ---
        print('Finished epoch:{} | Loss : {:.4f}'.format(
            epoch_idx + 1,
            np.mean(losses),
        ))
        torch.save(model.state_dict(), os.path.join(train_config['task_name'],
                                                    train_config['ckpt_name']))

    print('Done Training ...')


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='Arguments for ddpm training')
    parser.add_argument('--config', dest='config_path',
                        default='/content/config.yaml', type=str)
    args = parser.parse_args([]) # Modified this line to handle Colab kernel arguments
    train(args)

{'dataset_params': {'im_path': '/content/data/train/images'}, 'diffusion_params': {'num_timesteps': 1000, 'beta_start': 0.0001, 'beta_end': 0.02}, 'model_params': {'im_channels': 1, 'im_size': 28, 'down_channels': [32, 64, 128, 256], 'mid_channels': [256, 256, 128], 'down_sample': [True, True, False], 'time_emb_dim': 128, 'num_down_layers': 2, 'num_mid_layers': 2, 'num_up_layers': 2, 'num_heads': 4}, 'train_params': {'task_name': 'default', 'batch_size': 192, 'num_epochs': 40, 'num_samples': 100, 'num_grid_rows': 10, 'lr': 0.0001, 'ckpt_name': 'ddpm_ckpt.pth'}}


100%|██████████| 10/10 [00:16<00:00,  1.60s/it]


Found 60000 images for split train


/tmp/ipykernel_2933/774090045.py:64: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
  0%|          | 0/312 [00:00<?, ?it/s]/tmp/ipykernel_2933/774090045.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
W0316 06:18:33.051000 2933 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode
  0%|          | 1/312 [02:30<13:00:35, 150.60s/it]/tmp/ipykernel_2933/774090045.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
100%|██████████| 312/312 [03:38<00:00,  1.43it/s]


Finished epoch:1 | Loss : 0.1898


...

Finished epoch:38 | Loss : 0.0245


100%|██████████| 312/312 [01:09<00:00,  4.48it/s]


Finished epoch:39 | Loss : 0.0246


100%|██████████| 312/312 [01:09<00:00,  4.48it/s]


Finished epoch:40 | Loss : 0.0243
Done Training ...
